In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("superstore.db")
df = pd.read_sql_query("SELECT * FROM orders", conn)

# Convert the date column properly (handles M/D/YYYY automatically, no leading zeros needed)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')

# Save back to the same table, replacing the old one
df.to_sql("orders", conn, if_exists="replace", index=False)

conn.close()
print("Done — Order Date column converted and saved back to SQLite.")

Done — Order Date column converted and saved back to SQLite.


In [3]:
conn = sqlite3.connect("superstore.db")

In [4]:
query1 = """
SELECT
    "Order ID",
    Sales,
    Profit,
    ROUND(Profit * 1.0 / Sales, 4) AS profit_margin
FROM orders;
"""
pd.read_sql_query(query1, conn)

,Order ID,Sales,Profit,profit_margin
0,CA-2016-152156,261.9600,41.9136,0.1600
1,CA-2016-152156,731.9400,219.5820,0.3000
2,CA-2016-138688,14.6200,6.8714,0.4700
3,US-2015-108966,957.5775,-383.0310,-0.4000
4,US-2015-108966,22.3680,2.5164,0.1125
...,...,...,...,...
9989,CA-2014-110422,25.2480,4.1028,0.1625
9990,CA-2017-121258,91.9600,15.6332,0.1700
9991,CA-2017-121258,258.5760,19.3932,0.0750
9992,CA-2017-121258,29.6000,13.3200,0.4500


In [5]:
query2 = """
SELECT
    Region,
    SUM(Sales) AS total_sales,
    SUM(Profit) AS total_profit,
    ROUND(SUM(Profit) * 1.0 / SUM(Sales), 4) AS overall_profit_margin
FROM orders
GROUP BY Region
ORDER BY total_profit DESC;
"""
pd.read_sql_query(query2, conn)

,Region,total_sales,total_profit,overall_profit_margin
0,West,725457.8245,108418.4489,0.1494
1,East,678781.2400,91522.7800,0.1348
2,South,391721.9050,46749.4303,0.1193
3,Central,501239.8908,39706.3625,0.0792


In [6]:
query3 = """
SELECT
    Category,
    SUM(Sales) AS total_sales,
    SUM(Profit) AS total_profit,
    ROUND(SUM(Profit) * 1.0 / SUM(Sales), 4) AS overall_profit_margin
FROM orders
GROUP BY Category
ORDER BY total_profit DESC;
"""
pd.read_sql_query(query3, conn)

,Category,total_sales,total_profit,overall_profit_margin
0,Technology,836154.0330,145454.9481,0.1740
1,Office Supplies,719047.0320,122490.8008,0.1704
2,Furniture,741999.7953,18451.2728,0.0249


In [7]:
query4 = """
SELECT
    "Sub-Category",
    SUM(Sales) AS total_sales,
    SUM(Profit) AS total_profit,
    ROUND(SUM(Profit) * 1.0 / SUM(Sales), 4) AS overall_profit_margin
FROM orders
GROUP BY "Sub-Category"
ORDER BY overall_profit_margin ASC;
"""
pd.read_sql_query(query4, conn)

,Sub-Category,total_sales,total_profit,overall_profit_margin
0,Tables,206965.5320,-17725.4811,-0.0856
1,Bookcases,114879.9963,-3472.5560,-0.0302
2,Supplies,46673.5380,-1189.0995,-0.0255
3,Machines,189238.6310,3384.7569,0.0179
4,Chairs,328449.1030,26590.1663,0.0810
5,Storage,223843.6080,21278.8264,0.0951
6,Phones,330007.0540,44515.7306,0.1349
7,Furnishings,91705.1640,13059.1436,0.1424
8,Binders,203412.7330,30221.7633,0.1486
9,Appliances,107532.1610,18138.0054,0.1687


In [8]:
conn.execute("DROP VIEW IF EXISTS category_summary;")
conn.execute("""
CREATE VIEW category_summary AS
SELECT
    Category,
    SUM(Sales) AS total_sales,
    SUM(Profit) AS total_profit,
    ROUND(SUM(Profit) * 1.0 / SUM(Sales), 4) AS overall_profit_margin
FROM orders
GROUP BY Category;
""")

conn.execute("DROP VIEW IF EXISTS subcategory_summary;")
conn.execute("""
CREATE VIEW subcategory_summary AS
SELECT
    "Sub-Category",
    SUM(Sales) AS total_sales,
    SUM(Profit) AS total_profit,
    ROUND(SUM(Profit) * 1.0 / SUM(Sales), 4) AS overall_profit_margin
FROM orders
GROUP BY "Sub-Category";
""")

conn.execute("DROP VIEW IF EXISTS region_summary;")
conn.execute("""
CREATE VIEW region_summary AS
SELECT
    Region,
    SUM(Sales) AS total_sales,
    SUM(Profit) AS total_profit,
    ROUND(SUM(Profit) * 1.0 / SUM(Sales), 4) AS overall_profit_margin
FROM orders
GROUP BY Region;
""")

conn.commit()
print("Views created.")

Views created.


In [9]:
query5 = "SELECT * FROM subcategory_summary WHERE overall_profit_margin < 0;"
pd.read_sql_query(query5, conn)

,Sub-Category,total_sales,total_profit,overall_profit_margin
0,Bookcases,114879.9963,-3472.5560,-0.0302
1,Supplies,46673.5380,-1189.0995,-0.0255
2,Tables,206965.5320,-17725.4811,-0.0856


In [10]:
conn.execute("""
CREATE TABLE IF NOT EXISTS category_group (
    Category TEXT PRIMARY KEY,
    business_line TEXT
);
""")

conn.execute("""
INSERT OR REPLACE INTO category_group VALUES
    ('Technology', 'High Margin'),
    ('Office Supplies', 'High Margin'),
    ('Furniture', 'Low Margin');
""")
conn.commit()

query6 = """
SELECT
    o.Category,
    cg.business_line,
    SUM(o.Sales) AS total_sales,
    SUM(o.Profit) AS total_profit
FROM orders o
JOIN category_group cg ON o.Category = cg.Category
GROUP BY o.Category, cg.business_line;
"""
pd.read_sql_query(query6, conn)

,Category,business_line,total_sales,total_profit
0,Furniture,Low Margin,741999.7953,18451.2728
1,Office Supplies,High Margin,719047.0320,122490.8008
2,Technology,High Margin,836154.0330,145454.9481


In [11]:
query7 = """
SELECT
    strftime('%Y-%m', "Order Date") AS month,
    SUM(Sales) AS total_sales,
    SUM(Profit) AS total_profit
FROM orders
GROUP BY month
ORDER BY month;
"""
pd.read_sql_query(query7, conn)

,month,total_sales,total_profit
0,2014-01,14236.8950,2450.1907
1,2014-02,4519.8920,862.3084
2,2014-03,55691.0090,498.7299
3,2014-04,28295.3450,3488.8352
4,2014-05,23648.2870,2738.7096
5,2014-06,34595.1276,4976.5244
6,2014-07,33946.3930,-841.4826
7,2014-08,27909.4685,5318.1050
8,2014-09,81777.3508,8328.0994
9,2014-10,31453.3930,3448.2573


In [12]:
conn.close()

## Week 6 — Analytics Summary

**Top-line results:**
- Highest-performing region: West (highest sales and profit).
- Weakest-performing category by margin: Furniture (2.5% margin) despite high sales volume.
- Loss-making sub-categories: Tables (-8.6%), Bookcases (-3.0%), Supplies (-2.6%).
- Best month overall: December 2016 (highest sales and profit).
- Loss-making months: July 2014 and January 2015 — worth investigating (likely deep discounting).

**Key insight:** Sales volume does not guarantee profitability — Furniture generates significant revenue but is dragged down by consistently unprofitable sub-categories (especially Tables), while smaller categories like Technology are far more profitable per dollar sold.